In [1]:
import os
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, 
    recall_score, f1_score, matthews_corrcoef
)

# 1. Flexible Data Loading (.xlsx or .csv)
file_path = "UCI_Credit_Card.xlsx"  # Or "UCI_Credit_Card.csv"

if file_path.endswith('.csv'):
    df = pd.read_csv(file_path)
else:
    df = pd.read_excel(file_path, header=0)

# Clean column headers
df.columns = [str(c).strip() for c in df.columns]

# Target column identification & ID dropping
target_cols = [c for c in df.columns if 'default' in c.lower()]
if not target_cols:
    raise KeyError("No target column containing 'default' found in dataset.")

df.rename(columns={target_cols[0]: 'default'}, inplace=True)

if 'ID' in df.columns:
    df.drop(columns=['ID'], inplace=True)

X = df.drop(columns=['default'])
y = df['default']

# 2. Imputation for Missing Values
imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

# 3. Stratified Train-Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y, test_size=0.2, random_state=42, stratify=y
)

# Export test dataset for Streamlit upload requirement
# Save test data strictly as a clean CSV file
test_df = X_test.copy()
test_df['default'] = y_test
test_df.to_csv('test_data.csv', index=False, encoding='utf-8')
print("Saved clean test_data.csv!")

# 4. Required Pipeline Construction & Model Fitting Function
def get_trained_pipelines(X_train, y_train):
    models = {
        "Logistic Regression": LogisticRegression(max_iter=5000, random_state=42),
        "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
        "kNN": KNeighborsClassifier(n_neighbors=5),
        "Naive Bayes": GaussianNB(),
        "Random Forest (Ensemble)": RandomForestClassifier(
            n_estimators=200, max_depth=8, random_state=42
        ),
    }

    fitted_pipelines = {}
    for name, estimator in models.items():
        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("clf", estimator)
        ])
        pipe.fit(X_train, y_train)
        fitted_pipelines[name] = pipe

    return fitted_pipelines

# Fit Pipelines
fitted_pipelines = get_trained_pipelines(X_train, y_train)

# 5. Evaluate & Pickle All Artifacts into model/ Folder
os.makedirs('model', exist_ok=True)
joblib.dump(imputer, 'model/imputer.pkl')

file_name_map = {
    "Logistic Regression": "logistic_regression.pkl",
    "Decision Tree": "decision_tree.pkl",
    "kNN": "knn.pkl",
    "Naive Bayes": "naive_bayes.pkl",
    "Random Forest (Ensemble)": "random_forest_ensemble.pkl"
}

results = []
for name, pipe in fitted_pipelines.items():
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1] if hasattr(pipe, "predict_proba") else y_pred
    
    results.append({
        'ML Model Name': name,
        'Accuracy': round(accuracy_score(y_test, y_pred), 4),
        'AUC': round(roc_auc_score(y_test, y_proba), 4),
        'Precision': round(precision_score(y_test, y_pred, zero_division=0), 4),
        'Recall': round(recall_score(y_test, y_pred, zero_division=0), 4),
        'F1': round(f1_score(y_test, y_pred, zero_division=0), 4),
        'MCC': round(matthews_corrcoef(y_test, y_pred), 4)
    })
    
    joblib.dump(pipe, f'model/{file_name_map[name]}')

summary_df = pd.DataFrame(results)
summary_df.to_csv('results_summary.csv', index=False, encoding='utf-8')
print("Pipeline Training & Metric Evaluation Complete:")
print(summary_df)

Saved clean test_data.csv!
Pipeline Training & Metric Evaluation Complete:
              ML Model Name  Accuracy     AUC  Precision  Recall      F1  \
0       Logistic Regression    0.8077  0.7076     0.6868  0.2396  0.3553   
1             Decision Tree    0.8172  0.7418     0.6615  0.3549  0.4620   
2                       kNN    0.7928  0.7012     0.5487  0.3564  0.4322   
3               Naive Bayes    0.7525  0.7249     0.4515  0.5539  0.4975   
4  Random Forest (Ensemble)    0.8168  0.7731     0.6629  0.3497  0.4578   

      MCC  
0  0.3244  
1  0.3893  
2  0.3233  
3  0.3386  
4  0.3868  
